# Extract all the relevant data of all the scans

Wrestle with all the log files of all the scans.
We double-check all scanning and reconstruction parameters to look for inconsistencies to be corrected.
At the end we generate some helping files which we need for collaboration.

First set up the notebook with some imports and defaults.

In [1]:
# Load the python modules we need
import platform
import os
import glob
import pandas
import imageio
import numpy
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
import seaborn
import dask
import dask_image.imread
from dask.distributed import Client, LocalCluster
import skimage
from tqdm import notebook

In [2]:
# Load our own log file parsing code
from BrukerSkyScanLogfileRuminator.parsing_functions import *

In [3]:
# Set dask temporary folder
# Do this before creating a client: https://stackoverflow.com/a/62804525/323100
import tempfile
if 'Linux' in platform.system():
    # Check if me mounted the FastSSD, otherwise go to standard tmp file
    if os.path.exists(os.path.join(os.sep, 'media', 'habi', 'Fast_SSD')):
        tmp = os.path.join(os.sep, 'media', 'habi', 'Fast_SSD', 'tmp')
    else:
        tmp = tempfile.gettempdir()
elif 'Darwin' in platform.system():
    tmp = tempfile.gettempdir()
else:
    if 'anaklin' in platform.node():
        tmp = os.path.join('F:\\tmp')
    else:
        tmp = os.path.join('D:\\tmp')
dask.config.set({'temporary_directory': tmp})
print('Dask temporary files go to %s' % dask.config.get('temporary_directory'))

Dask temporary files go to /media/habi/Fast_SSD/tmp


In [4]:
from dask.distributed import Client
client = Client()

/home/habi/miniconda3/envs/sticklebacks/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 35705 instead
  warnings.warn(


In [5]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:35705/status,
Dashboard: http://127.0.0.1:35705/status,Workers: 8
Total threads: 48,Total memory: 377.19 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39669,Workers: 0
Dashboard: http://127.0.0.1:35705/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:41635,Total threads: 6
Dashboard: http://127.0.0.1:37737/status,Memory: 47.15 GiB
Nanny: tcp://127.0.0.1:34513,


In [6]:
print('You can see what DASK is doing at "http://localhost:%s/status"' % client.scheduler_info()['services']['dashboard'])

You can see what DASK is doing at "http://localhost:35705/status"


In [7]:
# Set up figure defaults
plt.rc('image', cmap='gray', interpolation='nearest')  # Display all images in b&w and with 'nearest' interpolation
plt.rcParams['figure.figsize'] = (16, 9)  # Size up figures a bit
plt.rcParams['figure.dpi'] = 200

In [8]:
# Setup scale bar defaults
plt.rcParams['scalebar.location'] = 'lower right'
plt.rcParams['scalebar.frameon'] = False
plt.rcParams['scalebar.color'] = 'white'

Since the (tomographic) data can reside on different drives we set a folder to use below

In [9]:
local = True
if local:
    # Load the log files from the repository subfolder.
    # Then we cannot 
    Root = os.path.join(os.getcwd(), 'logfiles')
else:
    Root = os.path.join('/home/habi/research_storage_ben/microCT_Stickleback/')
print('We are loading all the data from %s' % Root)

We are loading all the data from /home/habi/P/Documents/IEE/Sulser Sticklebacks/logfiles


We generate some output in this notebook.
To make all the data completely reproducible, save the output to a directory named according to the current `git` hash of the repository.

In [10]:
def get_git_hash():
    '''
    Get the current git hash from the repository.
    Based on http://stackoverflow.com/a/949391/323100 and
    http://stackoverflow.com/a/18283905/323100
    '''
    from subprocess import Popen, PIPE
    import os
    gitprocess = Popen(['git',
                        '--git-dir',
                        os.path.join(os.getcwd(), '.git'),
                        'rev-parse',
                        '--short',
                        '--verify',
                        'HEAD'],
                       stdout=PIPE)
    (output, _) = gitprocess.communicate()
    return output.strip().decode("utf-8")

In [11]:
# Make directory for output
OutPutDir = os.path.join(os.getcwd(), 'Output', get_git_hash())
print('We are saving all the output to %s' % OutPutDir)
os.makedirs(OutPutDir, exist_ok=True)

We are saving all the output to /home/habi/P/Documents/IEE/Sulser Sticklebacks/Output/417c678


Now that we are set up, actually start to load/ingest the data.

In [12]:
# Make us a dataframe for saving all that we need
Data = pandas.DataFrame()

In [13]:
# Get *all* log files, unsorted but fast
Data['LogFile'] = [os.path.join(root, name)
                   for root, dirs, files in os.walk(Root)
                   for name in files
                   if name.endswith((".log"))]

The notebook might not be running locally on our machines, but on Binder.
There, the user has no access to the log files, so we fail back to a local copy of them.
This also means that no reconstructions are available, und we thus cannot count them.
We thus set a variable which skips looking for parameters related to the reconstructions.

In [14]:
if not len(Data):
    # Our dataframe is empty.
    # We might be running on Binder, e.g. load the logfiles from the subfolder in this repository
    print(10 * ' -', 'CAVEAT', 10 * ' -')
    print('You are most probably running the notebook on binder.')
    print('And thus do not have access to the log files on the research storage')
    print('We are using a "local" copy of the data in the `logfiles` subfolder')
    print('This gives correct, but possibly outdated results...')
    print(10 * ' -', 'CAVEAT', 10 * ' -')
    # Change root folder
    Root = 'logfiles'
    # Load log files again
    Data['LogFile'] = [f for f in sorted(glob.glob(os.path.join(Root, '**', '*.log'),
                                                   recursive=True),
                                         key=os.path.getmtime)]
    running_on_binder = True
else:
    running_on_binder = False

In [15]:
# Get all folders
Data['Folder'] = [os.path.dirname(f) for f in Data['LogFile']]
Data['FolderShort'] = [f[len(Root)+1:] for f in Data['Folder']]

In [16]:
Data.Folder[0]

'/home/habi/P/Documents/IEE/Sulser Sticklebacks/logfiles/PilotScans/F13_F3-07_14-15/heads_10um_proj'

In [17]:
if not running_on_binder:
    # Check for samples which are not yet reconstructed
    for c, row in Data.iterrows():
        # Iterate over every 'proj' folder
        if 'proj' in row.Folder:
            if 'TScopy' not in row.Folder and 'PR' not in row.Folder:
                # If there's nothing with 'rec*' on the same level, then tell us
                if not glob.glob(row.Folder.replace('proj', 'rec')):
                    # print(glob.glob(row.Folder.replace('proj', 'rec')))
                    print('- %s is missing matching reconstructions' % row.LogFile[len(Root) + 1:])

- Sticklebucket_14/proj2/Sticklebucket_14~00.log is missing matching reconstructions
- Sticklebucket_15/proj2/Sticklebucket_15~00.log is missing matching reconstructions


In [18]:
# Get rid of all log files we do not want to talk about in the manuscript
for c, row in Data.iterrows():
    if 'PilotScans' in row.Folder:  # The pilot scans are mentioned, but not interesting
        Data.drop([c], inplace=True)    
    elif 'rec' not in row.Folder:  # We only want to look at 'rec' folders
        Data.drop([c], inplace=True)
    elif '_regions' in row.Folder:  # Do not look at the folders with the extracted regions
        Data.drop([c], inplace=True)
# Reset dataframe index
Data = Data.reset_index(drop=True)

In [19]:
Data.head()

,LogFile,Folder,FolderShort
0,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/15um_rec
1,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/17.5um_rec
2,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/18um_rec
3,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/19um_rec
4,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_B/rec


In [20]:
# Generate us some meaningful colums
Data['Bucket'] = [l[len(Root) + 1:].split(os.sep)[0] for l in Data['LogFile']]
Data['Scan'] = ['.'.join(l[len(Root) + 1:].split(os.sep)[1:-1]) for l in Data['LogFile']]

In [21]:
Data.head()

,LogFile,Folder,FolderShort,Bucket,Scan
0,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/15um_rec,BucketOfFish_A,15um_rec
1,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/17.5um_rec,BucketOfFish_A,17.5um_rec
2,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/18um_rec,BucketOfFish_A,18um_rec
3,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/19um_rec,BucketOfFish_A,19um_rec
4,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_B/rec,BucketOfFish_B,rec


In [22]:
# Get parameters related to scan from logfiles
Data['Scan date'] = [scandate(log) for log in Data['LogFile']]
Data['Scanner'] = [scanner(log) for log in Data['LogFile']]
Data['Voltage'] = [voltage(log) for log in Data['LogFile']]
Data['Current'] = [current(log) for log in Data['LogFile']]
Data['Filter'] = [whichfilter(log) for log in Data['LogFile']]
Data['Exposuretime'] = [exposuretime(log) for log in Data['LogFile']]
Data['Averaging'] = [averaging(log) for log in Data['LogFile']]
Data['Number of projections'] = [numproj(log) for log in Data['LogFile']]
Data['ProjectionSize'] = [projection_size(log) for log in Data['LogFile']]
Data['RotationStep'] = [rotationstep(log) for log in Data['LogFile']]
Data['ThreeSixty'] = [threesixtyscan(log) for log in Data['LogFile']]
Data['Voxelsize'] = [pixelsize(log) for log in Data['LogFile']]
Data['Duration'] = [duration(log) for log in Data['LogFile']]
Data['Stacks'] = [stacks(log) for log in Data['LogFile']]

In [23]:
# Get parameters related to reconstruction from logfiles
Data['Number of reconstructions'] = [slice_number(log) for log in Data['LogFile']]  # This is the number of reconstructions that NRecon wrote to disk and to the log file. *Not* necessarily the number that may be present on disk. In the InMice project we check if files on disk are the same as filenumber written to log file. But for this manuscript only reading from the log is good enough.
Data['ReconstructionSize'] = [reconstruction_size(log) for log in Data['LogFile']]
Data['Grayvalue'] = [reconstruction_grayvalue(log) for log in Data['LogFile']]
Data['RingartefactCorrection'] = [ringremoval(log) for log in Data['LogFile']]
Data['BeamHardeningCorrection'] = [beamhardening(log) for log in Data['LogFile']]
Data['ROI'] = [region_of_interest(log) for log in Data['LogFile']]
Data['NRecon'] = [nreconversion(log)[1] for log in Data['LogFile']]

In [24]:
Data.head()

,LogFile,Folder,FolderShort,Bucket,Scan,Scan date,Scanner,Voltage,Current,Filter,...,Voxelsize,Duration,Stacks,Number of reconstructions,ReconstructionSize,Grayvalue,RingartefactCorrection,BeamHardeningCorrection,ROI,NRecon
0,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/15um_rec,BucketOfFish_A,15um_rec,2023-11-11 12:03:24,SkyScan 2214,60.0,140.0,None,...,15.000176,11887.0,2,2177,"(3072, 3072)",0.064372,None,None,False,"(NRecon, 2.1.0.1)"
1,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/17.5um_rec,BucketOfFish_A,17.5um_rec,2023-11-11 19:31:27,SkyScan 2214,60.0,140.0,None,...,17.499569,11898.0,3,4206,"(3072, 3072)",0.041574,None,None,False,"(NRecon, 2.2.0.6)"
2,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/18um_rec,BucketOfFish_A,18um_rec,2023-11-10 22:04:16,SkyScan 2214,60.0,140.0,None,...,17.999448,11934.0,3,4094,"(1536, 1536)",0.040384,None,None,False,"(NRecon, 2.1.0.1)"
3,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/19um_rec,BucketOfFish_A,19um_rec,2023-11-12 23:58:37,SkyScan 2214,60.0,140.0,None,...,19.000720,10476.0,3,3867,"(3072, 3072)",0.056215,None,None,False,"(NRecon, 2.1.0.1)"
4,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_B/rec,BucketOfFish_B,rec,2023-11-28 19:38:15,SkyScan 2214,60.0,110.0,None,...,17.499569,13764.0,3,3715,"(3064, 3064)",0.040277,None,None,False,"(NRecon, 2.2.0.6)"


In [25]:
Data.NRecon.unique()

array([('NRecon', '2.1.0.1'), ('NRecon', '2.2.0.6')], dtype=object)

In [26]:
numpy.sort(Data.Duration.unique())

array([ 9826.,  9860.,  9953., 10000., 10219., 10262., 10264., 10295.,
       10378., 10476., 10591., 10596., 10816., 11292., 11398., 11407.,
       11420., 11502., 11644., 11647., 11849., 11870., 11887., 11898.,
       11934., 11979., 12055., 12183., 12721., 12789., 13179., 13727.,
       13764., 13824., 13830., 13909., 14962., 15436., 15631., 15654.,
       15878., 15938., 17859.])

In [27]:
# Quickly look at all the different values in our dataframe
DoNotWant = {'LogFile',
             'Folder',
             'FolderShort',
             'Bucket',
             'Scan date'}
for column in Data.columns:
    if column not in DoNotWant:
        print(column, numpy.sort(Data[column].unique()))

Scan ['15um_rec' '17.5um_rec' '18um_rec' '19um_rec' 'rec' 'rec3']
Scanner ['SkyScan 2214']
Voltage [49. 60.]
Current [104. 105. 106. 110. 120. 140. 159.]
Filter [None]
Exposuretime [ 706  720  737  744  787  874  882  914  946  960 1079 1089 1199 1344
 1350 1482 1500 1906]
Averaging [None]
Number of projections [3601]
ProjectionSize [(1536, 1944) (3066, 1944) (3072, 1944)]
RotationStep [0.1]
ThreeSixty [ True]
Voxelsize [15.000176 15.999933 16.499812 17.499569 17.999448 19.00072 ]
Duration [ 9826.  9860.  9953. 10000. 10219. 10262. 10264. 10295. 10378. 10476.
 10591. 10596. 10816. 11292. 11398. 11407. 11420. 11502. 11644. 11647.
 11849. 11870. 11887. 11898. 11934. 11979. 12055. 12183. 12721. 12789.
 13179. 13727. 13764. 13824. 13830. 13909. 14962. 15436. 15631. 15654.
 15878. 15938. 17859.]
Stacks [1 2 3 4]
Number of reconstructions [1743 1752 2177 3309 3366 3490 3543 3715 3739 3815 3851 3857 3867 4045
 4049 4078 4094 4137 4166 4199 4206 4219 4256 4263 4273 4280 4295 4327
 4352 4369 43

In [43]:
Data.NRecon[0][1]

'2.1.0.1'

In [28]:
# Sort dataframe by scan date
Data.sort_values(by=['Scan date'], inplace=True)
# Reset dataframe index
Data = Data.reset_index(drop=True)

In [29]:
Data.head()

,LogFile,Folder,FolderShort,Bucket,Scan,Scan date,Scanner,Voltage,Current,Filter,...,Voxelsize,Duration,Stacks,Number of reconstructions,ReconstructionSize,Grayvalue,RingartefactCorrection,BeamHardeningCorrection,ROI,NRecon
0,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/18um_rec,BucketOfFish_A,18um_rec,2023-11-10 22:04:16,SkyScan 2214,60.0,140.0,None,...,17.999448,11934.0,3,4094,"(1536, 1536)",0.040384,None,None,False,"(NRecon, 2.1.0.1)"
1,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/15um_rec,BucketOfFish_A,15um_rec,2023-11-11 12:03:24,SkyScan 2214,60.0,140.0,None,...,15.000176,11887.0,2,2177,"(3072, 3072)",0.064372,None,None,False,"(NRecon, 2.1.0.1)"
2,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/17.5um_rec,BucketOfFish_A,17.5um_rec,2023-11-11 19:31:27,SkyScan 2214,60.0,140.0,None,...,17.499569,11898.0,3,4206,"(3072, 3072)",0.041574,None,None,False,"(NRecon, 2.2.0.6)"
3,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_A/19um_rec,BucketOfFish_A,19um_rec,2023-11-12 23:58:37,SkyScan 2214,60.0,140.0,None,...,19.000720,10476.0,3,3867,"(3072, 3072)",0.056215,None,None,False,"(NRecon, 2.1.0.1)"
4,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,/home/habi/P/Documents/IEE/Sulser Sticklebacks...,BucketOfFish_B/rec,BucketOfFish_B,rec,2023-11-28 19:38:15,SkyScan 2214,60.0,110.0,None,...,17.499569,13764.0,3,3715,"(3064, 3064)",0.040277,None,None,False,"(NRecon, 2.2.0.6)"


In [30]:
# How many scans did we do?
print('We performed %s scans in total' % len(Data.Scan))

We performed 44 scans in total


In [31]:
Data['Total Duration'] = [st * stk for st, stk in zip(Data['Duration'], Data['Stacks'])]

In [32]:
# Show voxelsize per bucket
for c, vs in enumerate(sorted(Data.Voxelsize.unique())):
    print('-----vs: %s-----' % vs)
    print(Data[Data.Voxelsize == vs][['Bucket', 'Scan', 'Voxelsize']])

-----vs: 15.000176-----
            Bucket      Scan  Voxelsize
1   BucketOfFish_A  15um_rec  15.000176
6   BucketOfFish_D       rec  15.000176
7   BucketOfFish_E       rec  15.000176
8   BucketOfFish_F       rec  15.000176
9   BucketOfFish_G       rec  15.000176
10  BucketOfFish_H       rec  15.000176
17  BucketOfFish_N       rec  15.000176
22  BucketOfFish_S       rec  15.000176
24  BucketOfFish_U       rec  15.000176
-----vs: 15.999933-----
              Bucket  Scan  Voxelsize
12    BucketOfFish_J   rec  15.999933
15    BucketOfFish_L   rec  15.999933
16    BucketOfFish_M   rec  15.999933
18    BucketOfFish_O   rec  15.999933
19    BucketOfFish_P   rec  15.999933
23    BucketOfFish_T   rec  15.999933
25  Sticklebucket_01   rec  15.999933
26  Sticklebucket_02   rec  15.999933
27  Sticklebucket_03   rec  15.999933
28  Sticklebucket_04   rec  15.999933
29  Sticklebucket_05   rec  15.999933
30  Sticklebucket_06   rec  15.999933
31  Sticklebucket_07   rec  15.999933
32  Sticklebucket_08

In [33]:
Data.Filter.unique()

array([None], dtype=object)

In [34]:
sorted(Data.Voltage.unique())

[np.float64(49.0), np.float64(60.0)]

In [35]:
sorted(Data.Current.unique())

[np.float64(104.0),
 np.float64(105.0),
 np.float64(106.0),
 np.float64(110.0),
 np.float64(120.0),
 np.float64(140.0),
 np.float64(159.0)]

In [36]:
sorted(Data.RingartefactCorrection.unique())

[None]

In [37]:
sorted(Data.BeamHardeningCorrection.unique())

[None]

In [38]:
# Get an overview of the total scaning time
# Nice output based on https://stackoverflow.com/a/8907407/323100
total_seconds = int(Data['Total Duration'].sum())
hours, remainder = divmod(total_seconds, 60 * 60)
minutes, seconds = divmod(remainder, 60)
print('In total, we scanned for %s hours and %s minutes)' % (hours, minutes))
for machine in Data['Scanner'].unique():
    total_seconds = int(Data[Data['Scanner'] == machine]['Total Duration'].sum())
    hours, remainder = divmod(total_seconds, 60 * 60)
    minutes, seconds = divmod(remainder, 60)
    print('\t - Of these, we scanned %s hours and %s minutes on the %s,'
          ' for %s scans' % (hours,
                             minutes,
                             machine,
                             len(Data[Data['Scanner'] == machine])))

In total, we scanned for 444 hours and 6 minutes)
	 - Of these, we scanned 444 hours and 6 minutes on the SkyScan 2214, for 44 scans


In [39]:
# We scanned six fish per scan, so approximate how many fish we scanned
print('We scanned %0.f fishes (%s scans at 6 fish per scan)' % (len(Data.Bucket.unique()) * 6, len(Data)))

We scanned 228 fishes (44 scans at 6 fish per scan)


In [40]:
Data.columns

Index(['LogFile', 'Folder', 'FolderShort', 'Bucket', 'Scan', 'Scan date',
       'Scanner', 'Voltage', 'Current', 'Filter', 'Exposuretime', 'Averaging',
       'Number of projections', 'ProjectionSize', 'RotationStep', 'ThreeSixty',
       'Voxelsize', 'Duration', 'Stacks', 'Number of reconstructions',
       'ReconstructionSize', 'Grayvalue', 'RingartefactCorrection',
       'BeamHardeningCorrection', 'ROI', 'NRecon', 'Total Duration'],
      dtype='object')

In [41]:
# Save 'data' file for https://github.com/habi/sticklebacks-manuscript
# Since the manuscript is in a subfolder, we can simply write the output there
if not running_on_binder:
    Data[['FolderShort', 'Bucket', 'Scan', 'Scanner', 'Scan date',
          'Voxelsize', 'Voltage', 'Current',
          'Filter', 'Exposuretime', 'Averaging',
          'Number of projections', 'ProjectionSize', 'RotationStep', 'ThreeSixty',
          'Duration', 'Stacks', 'Total Duration',
          'Number of reconstructions', 'ReconstructionSize', 'ROI',
          'RingartefactCorrection', 'BeamHardeningCorrection', 'Grayvalue', 'NRecon'
          ]].to_csv(os.path.join('manuscript', 'content', 'data', 'ScanningDetails.csv'),
                    index=False,
                    header=['Folder', 'Bucket', 'Scan', 'Scanner', 'Scan date',
                            'Voxelsize [μm]', 'Source voltage [kV]', 'Source current [μA]',
                            'Filter', 'Exposure time [ms]', 'Frame averaging',
                            'Number of projections', 'Projection size [px]', 'Rotation step [°]', '360° scan',
                            'Scan duration [s]', 'Stacked scans', 'Total scan duration [s]',                                     
                            'Number of reconstructions', 'Reconstruction size [px]', 'Region of interest for reconstruction',
                            'Ring removal correction', 'Beam hardening correction', 'Gray value mapping', 'NRecon version'
                            ])
print('Saved CSV file with all relevant scanning and reconstruction parameters to',
      os.path.join('manuscript', 'content', 'data', 'ScanningDetails.csv'),
      'for using as supplementary material in the manuscript')

Saved CSV file with all relevant scanning and reconstruction parameters to manuscript/content/data/ScanningDetails.csv for using as supplementary material in the manuscript
